# 12. Contextual RAG - 上下文增强检索

**复杂度：** ⭐⭐⭐

## 概述

**Contextual RAG（上下文增强检索）**是 Anthropic 提出的一种技术，通过为每个文本块添加关于其在更大文档中作用的上下文信息来提高检索质量。

### 问题所在

在标准 RAG 中，文档块会失去其上下文：
- 块被孤立地嵌入
- 引用（"这"、"这些方法"、"上述"）变得模糊不清
- 主题边界不清晰
- 由于缺少上下文，检索可能会遗漏相关块

### 解决方案

Contextual RAG 在每个块前面添加：
1. **文档级摘要**：整个文档的主题是什么
2. **块级上下文**：这个特定块与文档的关系如何

```
标准块："该函数接受两个参数..."
上下文块："[本节描述了用户管理 API 中认证模块的
           登录函数。] 
           该函数接受两个参数..."
```

### 处理流程

```
文档 → 分割成块 → 对每个块：
    1. 生成文档摘要（每个文档一次）
    2. 生成块上下文（每个块）
    3. 将上下文前置到块中
    4. 嵌入上下文块
→ 存储到向量数据库 → 检索 → 生成答案
```

### 适用场景

✅ **适合用于：**
- 包含许多章节的长文档
- 带有交叉引用的技术文档
- 上下文很重要的文档（法律、医疗、学术）
- 提高模糊查询的精确度

❌ **不适合用于：**
- 简短、自包含的文档
- 实时应用（上下文生成会增加延迟）
- 成本敏感的应用（额外的 LLM 调用）

### 权衡取舍

**优点：**
- ✅ 更好的检索精确度
- ✅ 处理模糊引用
- ✅ 保持文档结构感知
- ✅ 一次性预处理成本

**缺点：**
- ❌ 更高的索引成本（每个块都需要 LLM 调用）
- ❌ 更大的嵌入向量（上下文 + 内容）
- ❌ 更复杂的预处理管道
- ❌ 索引时间更长

---

## 实现

让我们逐步构建 Contextual RAG。

## 1. 设置和导入

In [3]:
import sys
import time
from pathlib import Path

# Add parent directory to path for imports
sys.path.append(str(Path("../..").resolve()))

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from shared.config import (
    verify_api_key,
    DEFAULT_MODEL,
    DEFAULT_TEMPERATURE,
    HF_EMBEDDING_MODEL,
    VECTOR_STORE_DIR,
)
from shared.loaders import load_and_split
from shared.prompts import (
    DOCUMENT_SUMMARY_PROMPT,
    CONTEXTUAL_CHUNK_PROMPT,
    CONTEXTUAL_RAG_ANSWER_PROMPT,
    RAG_PROMPT_TEMPLATE,
)
from shared.utils import (
    format_docs,
    print_section_header,
    load_vector_store,
    save_vector_store,
)

# Verify API key
verify_api_key()

print("✓ All imports successful")
print(f"✓ Using model: {DEFAULT_MODEL}")
print(f"✓ Using embeddings: {HF_EMBEDDING_MODEL}")

OK deepseek API key: LOADED
  Preview: sk-0501...20ae
  Base URL: https://api.deepseek.com
✓ All imports successful
✓ Using model: deepseek-v4-flash
✓ Using embeddings: sentence-transformers/all-MiniLM-L6-v2


## 2. 加载和准备文档

In [4]:
print_section_header("Loading Documents")

# Load and split documents (returns tuple: original_docs, chunks)
_, docs = load_and_split(
    chunk_size=1000,
    chunk_overlap=200,
)

print(f"\n✓ Loaded {len(docs)} chunks")
print(f"✓ Average chunk size: {sum(len(d.page_content) for d in docs) / len(docs):.0f} chars")

# Show example chunk
print("\n" + "=" * 80)
print("Example chunk (standard):")
print("=" * 80)
print(docs[5].page_content[:300] + "...")


LOADING DOCUMENTS

Loading 4 documents from web...
  - https://python.langchain.com/docs/use_cases/question_answering/
  - https://python.langchain.com/docs/modules/data_connection/retrievers/
  - https://python.langchain.com/docs/modules/model_io/llms/
  - https://python.langchain.com/docs/use_cases/chatbots/
✓ Loaded 4 documents
✓ Added custom metadata to all documents
Splitting documents...
  - Chunk size: 1000
  - Chunk overlap: 200
✓ Created 165 chunks

  Sample chunk:
    - Length: 995 chars
    - Source: https://python.langchain.com/docs/use_cases/question_answering/
    - Preview: Build a RAG agent with LangChain - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. B...

✓ Loaded 165 chunks
✓ Average chunk size: 756 chars

Example chunk (standard):
Expand for full code snippetimport bs4
from langchain.agents import AgentState, create_agent
from langchain_community.document_loaders import WebBaseLoader
from langchai

## 3. 按文档分组块

我们需要按源文档对块进行分组，以便生成文档级别的摘要。

In [5]:
from collections import defaultdict

print_section_header("Grouping Chunks by Document")

# Group chunks by source document
docs_by_source = defaultdict(list)
for doc in docs:
    source = doc.metadata.get("source", "unknown")
    docs_by_source[source].append(doc)

print(f"\n✓ Found {len(docs_by_source)} unique source documents")
print("\nChunks per source:")
for source, chunks in list(docs_by_source.items())[:5]:
    print(f"  • {source}: {len(chunks)} chunks")


GROUPING CHUNKS BY DOCUMENT


✓ Found 4 unique source documents

Chunks per source:
  • https://python.langchain.com/docs/use_cases/question_answering/: 46 chunks
  • https://python.langchain.com/docs/modules/data_connection/retrievers/: 7 chunks
  • https://python.langchain.com/docs/modules/model_io/llms/: 66 chunks
  • https://python.langchain.com/docs/use_cases/chatbots/: 46 chunks


## 4. 生成文档摘要

对于每个源文档，我们将生成一个捕获其主要目的和主题的摘要。

In [6]:
print_section_header("Generating Document Summaries")

# Initialize LLM for summarization
llm = ChatOpenAI(
    model=DEFAULT_MODEL,
    temperature=DEFAULT_TEMPERATURE,
)

# Create summarization chain
summary_chain = DOCUMENT_SUMMARY_PROMPT | llm | StrOutputParser()

# Generate summaries for each source document
doc_summaries = {}
print("\nGenerating summaries...")

for i, (source, chunks) in enumerate(docs_by_source.items(), 1):
    # Combine first few chunks to represent the document
    # (using all chunks would be too long and expensive)
    doc_text = "\n\n".join([chunk.page_content for chunk in chunks[:3]])
    
    # Generate summary
    summary = summary_chain.invoke({"document": doc_text})
    doc_summaries[source] = summary
    
    print(f"  [{i}/{len(docs_by_source)}] {source[:50]}...")
    
    # Rate limiting to avoid API throttling
    if i < len(docs_by_source):
        time.sleep(0.5)

print(f"\n✓ Generated {len(doc_summaries)} document summaries")

# Show example summary
example_source = list(doc_summaries.keys())[0]
print("\n" + "=" * 80)
print(f"Example summary for: {example_source}")
print("=" * 80)
print(doc_summaries[example_source])


GENERATING DOCUMENT SUMMARIES


Generating summaries...
  [1/4] https://python.langchain.com/docs/use_cases/questi...
  [2/4] https://python.langchain.com/docs/modules/data_con...
  [3/4] https://python.langchain.com/docs/modules/model_io...
  [4/4] https://python.langchain.com/docs/use_cases/chatbo...

✓ Generated 4 document summaries

Example summary for: https://python.langchain.com/docs/use_cases/question_answering/
This document is a tutorial from the LangChain documentation that explains how to build a Retrieval Augmented Generation (RAG) agent for question-answering over unstructured text. It covers the core steps of indexing (loading, splitting, and storing documents) and retrieval/generation, including RAG chains and security considerations. The guide provides a practical walkthrough for creating a simple Q&A chatbot using LangChain.


## 5. 生成上下文块

现在我们将用上下文信息增强每个块。

In [7]:
print_section_header("Generating Contextual Chunks")

# Create contextualization chain
context_chain = CONTEXTUAL_CHUNK_PROMPT | llm | StrOutputParser()

# Generate contextual chunks
contextual_docs = []
print(f"\nProcessing {len(docs)} chunks...")
print("(This may take a few minutes)\n")

for i, doc in enumerate(docs, 1):
    source = doc.metadata.get("source", "unknown")
    doc_summary = doc_summaries.get(source, "No summary available")
    
    # Generate contextual description
    context = context_chain.invoke({
        "doc_summary": doc_summary,
        "chunk": doc.page_content[:500],  # Limit chunk size for context generation
    })
    
    # Create new document with context prepended
    contextual_content = f"[Context: {context}]\n\n{doc.page_content}"
    
    contextual_doc = Document(
        page_content=contextual_content,
        metadata={
            **doc.metadata,
            "context": context,
            "original_content": doc.page_content,
        },
    )
    contextual_docs.append(contextual_doc)
    
    # Progress indicator
    if i % 10 == 0:
        print(f"  Processed {i}/{len(docs)} chunks...")
    
    # Rate limiting
    if i < len(docs):
        time.sleep(0.3)

print(f"\n✓ Generated {len(contextual_docs)} contextual chunks")


GENERATING CONTEXTUAL CHUNKS


Processing 165 chunks...
(This may take a few minutes)

  Processed 10/165 chunks...
  Processed 20/165 chunks...
  Processed 30/165 chunks...
  Processed 40/165 chunks...
  Processed 50/165 chunks...
  Processed 60/165 chunks...
  Processed 70/165 chunks...
  Processed 80/165 chunks...
  Processed 90/165 chunks...
  Processed 100/165 chunks...
  Processed 110/165 chunks...
  Processed 120/165 chunks...
  Processed 130/165 chunks...
  Processed 140/165 chunks...
  Processed 150/165 chunks...
  Processed 160/165 chunks...

✓ Generated 165 contextual chunks


## 6. 比较标准块与上下文块

In [8]:
print_section_header("Comparing Standard vs Contextual Chunks")

# Show example comparison
example_idx = 5

print("\n" + "=" * 80)
print("STANDARD CHUNK:")
print("=" * 80)
print(docs[example_idx].page_content[:400] + "...")

print("\n" + "=" * 80)
print("CONTEXTUAL CHUNK:")
print("=" * 80)
print(contextual_docs[example_idx].page_content[:500] + "...")

print("\n" + "=" * 80)
print("STATISTICS:")
print("=" * 80)
avg_context_len = sum(
    len(doc.metadata.get("context", "")) for doc in contextual_docs
) / len(contextual_docs)
print(f"Average context length: {avg_context_len:.0f} characters")
print(f"Context overhead: {avg_context_len / 1000 * 100:.1f}% (of 1000 char chunks)")


COMPARING STANDARD VS CONTEXTUAL CHUNKS


STANDARD CHUNK:
Expand for full code snippetimport bs4
from langchain.agents import AgentState, create_agent
from langchain_community.document_loaders import WebBaseLoader
from langchain.messages import MessageLikeRepresentation
from langchain_text_splitters import RecursiveCharacterTextSplitter...

CONTEXTUAL CHUNK:
[Context: This code imports the essential libraries for the document loading and splitting phase of the RAG pipeline, including tools to fetch web pages, define agent state, and split text into manageable chunks for indexing.]

Expand for full code snippetimport bs4
from langchain.agents import AgentState, create_agent
from langchain_community.document_loaders import WebBaseLoader
from langchain.messages import MessageLikeRepresentation
from langchain_text_splitters import RecursiveCharacterText...

STATISTICS:
Average context length: 222 characters
Context overhead: 22.2% (of 1000 char chunks)


## 7. 创建向量存储

我们将创建两个向量存储来进行比较：
1. 标准 RAG（无上下文）
2. Contextual RAG（带上下文）

In [9]:
from langchain_community.vectorstores import FAISS

print_section_header("Creating Vector Stores")

# Initialize embeddings
embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

# Try to load existing vector stores
standard_store_path = VECTOR_STORE_DIR / "contextual_rag_standard"
contextual_store_path = VECTOR_STORE_DIR / "contextual_rag_contextual"

print("\nChecking for existing vector stores...")

# Standard vector store
vectorstore_standard = load_vector_store(
    standard_store_path,
    embeddings,
)

if vectorstore_standard is None:
    print("\nCreating standard vector store...")
    vectorstore_standard = FAISS.from_documents(docs, embeddings)
    save_vector_store(vectorstore_standard, standard_store_path)
    print("✓ Standard vector store created and saved")
else:
    print("✓ Loaded existing standard vector store")

# Contextual vector store
vectorstore_contextual = load_vector_store(
    contextual_store_path,
    embeddings,
)

if vectorstore_contextual is None:
    print("\nCreating contextual vector store...")
    vectorstore_contextual = FAISS.from_documents(contextual_docs, embeddings)
    save_vector_store(vectorstore_contextual, contextual_store_path)
    print("✓ Contextual vector store created and saved")
else:
    print("✓ Loaded existing contextual vector store")

print("\n✓ Both vector stores ready")


CREATING VECTOR STORES



NameError: name 'OPENAI_EMBEDDING_MODEL' is not defined

## 8. 构建 RAG 链

In [ ]:
print_section_header("Building RAG Chains")

# Create retrievers
retriever_standard = vectorstore_standard.as_retriever(
    search_kwargs={"k": 4}
)
retriever_contextual = vectorstore_contextual.as_retriever(
    search_kwargs={"k": 4}
)

# Standard RAG chain
chain_standard = (
    {"context": retriever_standard | format_docs, "input": RunnablePassthrough()}
    | RAG_PROMPT_TEMPLATE
    | llm
    | StrOutputParser()
)

# Contextual RAG chain
chain_contextual = (
    {"context": retriever_contextual | format_docs, "input": RunnablePassthrough()}
    | CONTEXTUAL_RAG_ANSWER_PROMPT
    | llm
    | StrOutputParser()
)

print("✓ Standard RAG chain created")
print("✓ Contextual RAG chain created")

## 9. 测试和比较

让我们用那些受益于上下文的查询来测试这两种方法。

In [ ]:
print_section_header("Testing Queries")

# Test queries that benefit from context
test_queries = [
    "How do I use LCEL to build chains?",
    "What are the different types of memory in LangChain?",
    "Explain the role of retrievers in RAG applications",
]

for i, query in enumerate(test_queries, 1):
    print("\n" + "=" * 80)
    print(f"Query {i}: {query}")
    print("=" * 80)
    
    # Standard RAG
    print("\n[STANDARD RAG]")
    print("-" * 80)
    start_time = time.time()
    response_standard = chain_standard.invoke(query)
    time_standard = time.time() - start_time
    print(response_standard)
    print(f"\n⏱️  Time: {time_standard:.2f}s")
    
    # Contextual RAG
    print("\n[CONTEXTUAL RAG]")
    print("-" * 80)
    start_time = time.time()
    response_contextual = chain_contextual.invoke(query)
    time_contextual = time.time() - start_time
    print(response_contextual)
    print(f"\n⏱️  Time: {time_contextual:.2f}s")
    
    # Comparison
    print("\n" + "-" * 80)
    print("COMPARISON:")
    print(f"  • Latency difference: {abs(time_contextual - time_standard):.2f}s")
    print(f"  • Response length difference: {len(response_contextual) - len(response_standard)} chars")

## 10. 检索质量比较

让我们检查每种方法检索到的文档。

In [ ]:
print_section_header("Retrieval Quality Analysis")

test_query = "What are the different types of memory in LangChain?"

print(f"\nQuery: {test_query}")
print("\n" + "=" * 80)

# Standard retrieval
docs_standard = retriever_standard.invoke(test_query)
print("\n[STANDARD RETRIEVAL]")
print("-" * 80)
for i, doc in enumerate(docs_standard, 1):
    print(f"\nDocument {i}:")
    print(f"Source: {doc.metadata.get('source', 'unknown')[:60]}")
    print(f"Preview: {doc.page_content[:200]}...")

# Contextual retrieval
docs_contextual = retriever_contextual.invoke(test_query)
print("\n" + "=" * 80)
print("\n[CONTEXTUAL RETRIEVAL]")
print("-" * 80)
for i, doc in enumerate(docs_contextual, 1):
    print(f"\nDocument {i}:")
    print(f"Source: {doc.metadata.get('source', 'unknown')[:60]}")
    print(f"Context: {doc.metadata.get('context', 'N/A')[:150]}...")
    print(f"Preview: {doc.metadata.get('original_content', doc.page_content)[:200]}...")

## 11. 性能指标

In [ ]:
print_section_header("Performance Metrics")

# Indexing costs
num_documents = len(docs)
num_summaries = len(doc_summaries)
num_context_calls = len(contextual_docs)

print("\nINDEXING COSTS:")
print("-" * 80)
print(f"Documents processed: {num_documents}")
print(f"Document summaries generated: {num_summaries}")
print(f"Chunk contexts generated: {num_context_calls}")
print(f"Total LLM calls for contextualization: {num_summaries + num_context_calls}")
print(f"\nEstimated additional indexing time: ~{(num_summaries + num_context_calls) * 0.5 / 60:.1f} minutes")

# Storage costs
avg_standard_len = sum(len(doc.page_content) for doc in docs) / len(docs)
avg_contextual_len = sum(len(doc.page_content) for doc in contextual_docs) / len(contextual_docs)
overhead = (avg_contextual_len - avg_standard_len) / avg_standard_len * 100

print("\nSTORAGE OVERHEAD:")
print("-" * 80)
print(f"Average standard chunk: {avg_standard_len:.0f} chars")
print(f"Average contextual chunk: {avg_contextual_len:.0f} chars")
print(f"Overhead: {overhead:.1f}%")

# Query costs
print("\nQUERY COSTS:")
print("-" * 80)
print("Standard RAG: k retrievals + 1 generation")
print("Contextual RAG: k retrievals + 1 generation (same as standard)")
print("\n✓ No additional query-time cost!")

## 12. 关键要点

### 总结

**Contextual RAG** 通过用上下文信息增强块来提高检索质量：
- 文档级摘要提供高层上下文
- 块级描述阐明每个块的作用
- 更好地处理模糊引用和交叉引用

### 成本效益分析

| 方面 | 影响 | 说明 |
|--------|--------|-------|
| **索引时间** | ❌ +50-100% | 一次性成本 |
| **索引成本** | ❌ +$X | 每个块的 LLM 调用 |
| **存储** | ❌ +15-30% | 更大的嵌入向量 |
| **查询时间** | ✅ 相同 | 无运行时开销 |
| **查询成本** | ✅ 相同 | 无额外调用 |
| **检索质量** | ✅ 更好 | 提高精确度 |
| **答案质量** | ✅ 更好 | 更具上下文感知 |

### 最佳实践

1. **用于长文档**：当文档长且复杂时最有益
2. **批处理**：批量生成上下文以降低成本
3. **缓存摘要**：存储文档摘要以避免重新生成
4. **平衡上下文长度**：保持上下文简洁（1-2 句话）
5. **质量检查**：手动审查生成的上下文样本

### 何时使用

选择 **Contextual RAG** 当：
- ✅ 文档质量比成本更重要
- ✅ 处理技术性或复杂性文档
- ✅ 用户提出模糊或依赖上下文的问题
- ✅ 可接受一次性索引成本

坚持使用 **标准 RAG** 当：
- ✅ 文档简短且自包含
- ✅ 成本优化至关重要
- ✅ 需要实时索引
- ✅ 偏好更简单的实现

### 下一步

- **与其他技术结合**：上下文块与重排序配合良好
- **实验上下文生成**：尝试不同的提示策略
- **衡量影响**：使用 RAGAS 指标量化改进
- **优化成本**：使用更便宜的模型进行上下文生成

---

**复杂度评级：** ⭐⭐⭐（中等 - 概念简单，一些实现开销）

**生产就绪度：** ⭐⭐⭐⭐（高 - 经过验证的技术，权衡较小）

继续学习 **13_fusion_rag.ipynb** 了解使用互惠秩融合的 RAG-Fusion！